In [1]:
# print current working directory
import os
print(os.getcwd())

c:\Rashmi\python-medallion-coding-assignment-1\payment-datalake\notebooks


In [2]:
import os
import duckdb

# Determine output directory path based on where the notebook is run
base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
output_dir = os.path.join(base_dir, "output")
print(f"Using output directory: {os.path.abspath(output_dir)}")

conn = duckdb.connect()

# 1. Query Gold daily summary
print("--- Gold: Daily Payment Summary ---")
try:
    df_gold_summary = conn.execute(
        f"SELECT * FROM read_parquet('{output_dir}/gold/daily_payment_summary/**/*.parquet') ORDER BY event_date, merchant_id"
    ).df()
    display(df_gold_summary)
except Exception as e:
    print("Error reading Gold daily summary:", e)

# 2. Query Bronze (all partitions)
print("\n--- Bronze: Row Counts by Date ---")
try:
    df_bronze = conn.execute(
        f"SELECT event_date, COUNT(*) as rows FROM read_parquet('{output_dir}/bronze/**/*.parquet') GROUP BY event_date ORDER BY event_date"
    ).df()
    display(df_bronze)
except Exception as e:
    print("Error reading Bronze parquet:", e)

# 3. Check merchant performance rolling window
print("\n--- Gold: Merchant Performance 7d ---")
try:
    df_merchant_perf = conn.execute(
        f"SELECT * FROM read_parquet('{output_dir}/gold/merchant_performance_7d/**/*.parquet')"
    ).df()
    display(df_merchant_perf)
except Exception as e:
    print("Error reading Merchant performance:", e)


Using output directory: c:\Rashmi\python-medallion-coding-assignment-1\payment-datalake\output
--- Gold: Daily Payment Summary ---


,merchant_id,merchant_name,merchant_category,currency,status,transaction_count,total_amount,avg_amount,max_amount,approval_rate,event_date
0,M001,TechGadgets Ltd,RETAIL,GBP,APPROVED,2,428.99,214.495,299.00,0.5000,2024-01-15
1,M001,TechGadgets Ltd,RETAIL,GBP,DECLINED,2,141.99,70.995,89.99,0.5000,2024-01-15
2,M002,FastFood Express,FOOD,USD,APPROVED,2,68.25,34.125,45.50,0.6667,2024-01-15
3,M002,FastFood Express,FOOD,USD,DECLINED,1,78.30,78.300,78.30,0.6667,2024-01-15
4,M003,GlobalTravel Co,TRAVEL,SGD,APPROVED,2,1195.00,597.500,875.00,NaN,2024-01-15
5,M004,EduLearn Platform,EDUCATION,AUD,APPROVED,1,199.00,199.000,199.00,1.0000,2024-01-15
6,M004,EduLearn Platform,EDUCATION,AUD,PENDING,1,399.00,399.000,399.00,1.0000,2024-01-15
7,M005,HealthPlus Pharmacy,HEALTH,GBP,APPROVED,2,111.75,55.875,64.50,1.0000,2024-01-15
8,M006,AutoParts Direct,RETAIL,USD,APPROVED,1,415.00,415.000,415.00,1.0000,2024-01-15
9,M006,AutoParts Direct,RETAIL,USD,REVERSED,1,156.00,156.000,156.00,1.0000,2024-01-15



--- Bronze: Row Counts by Date ---


,event_date,rows
0,2024-01-15,17
1,2024-01-16,15



--- Gold: Merchant Performance 7d ---


,merchant_id,merchant_name,total_transactions_7d,total_approved_amount_7d,approval_rate_7d,reversal_rate_7d,active_days_7d,snapshot_date
0,M001,TechGadgets Ltd,4,428.99,0.5000,NaN,1,2024-01-15
1,M002,FastFood Express,3,68.25,0.6667,NaN,1,2024-01-15
2,M003,GlobalTravel Co,2,1195.00,NaN,NaN,1,2024-01-15
3,M004,EduLearn Platform,2,199.00,1.0000,NaN,1,2024-01-15
4,M005,HealthPlus Pharmacy,2,111.75,1.0000,NaN,1,2024-01-15
5,M006,AutoParts Direct,2,415.00,1.0000,0.5000,1,2024-01-15
6,M007,StreamFlix Media,1,12.99,1.0000,NaN,1,2024-01-15
7,M008,QuickBite Delivery,1,55.00,NaN,NaN,1,2024-01-15
8,M001,TechGadgets Ltd,7,713.98,0.6667,0.1429,2,2024-01-16
9,M002,FastFood Express,5,102.15,0.6000,NaN,2,2024-01-16
